In [4]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import dash_leaflet as dl
from dash import dcc, html
import plotly.express as px
from dash import dash_table
from dash.dependencies import Input, Output, State
import base64
JupyterDash.infer_jupyter_proxy_config()

# Configure OS routines
import os

# Configure the plotting routines
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import re # regex import so I can parse through data


#### FIX ME #####
# change animal_shelter and AnimalShelter to match your CRUD Python module file name and class name
from CRUD_Python_Module import AnimalShelter

###########################
# Data Manipulation / Model
###########################
# FIX ME update with your username and password and CRUD Python module name

username = "aacuser"
password = "aacpassword" # use my password and user

# Connect to database via CRUD Module
db = AnimalShelter(username, password)

# class read method must support return of list object and accept projection json input
# sending the read method an empty document requests all documents be returned
df = pd.DataFrame.from_records(db.read({}))

# MongoDB v5+ is going to return the '_id' column and that is going to have an 
# invlaid object type of 'ObjectID' - which will cause the data_table to crash - so we remove
# it in the dataframe here. The df.drop command allows us to drop the column. If we do not set
# inplace=True - it will reeturn a new dataframe that does not contain the dropped column(s)
df.drop(columns=['_id'],inplace=True)

## Debug
# print(len(df.to_dict(orient='records')))
# print(df.columns)


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

#FIX ME Add in Grazioso Salvare’s logo
image_filename = 'Grazioso Salvare Logo.png' # replace with your own image
encoded_image = base64.b64encode(open(image_filename, 'rb').read())

#FIX ME Place the HTML image tag in the line below into the app.layout code according to your design
#FIX ME Also remember to include a unique identifier such as your name or date
#html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))

app.layout = html.Div([
    # html.Div(id='hidden-div', style={'display':'none'}),
    html.Center(html.Img(src='data:image/png;base64,{}'.format(encoded_image.decode()))), #html image tag
    html.Center(html.B(html.H1('CS-340 Dashboard - Neo Huynh'))), # add my own unifier
    html.Hr(),
    html.Div([
        
#FIXME Add in code for the interactive filtering options. For example, Radio buttons, drop down, checkboxes, etc.
        html.Label('Select Rescue Type'),
        dcc.Dropdown( 
            id='filter-type',
            options = [
                {'label' : 'All', 'value' : 'All'},
                {'label' : 'Water Rescue', 'value' : 'Water'},
                {'label' : 'Mountain or Wilderness Rescue', 'value' : 'Mountain'},
                {'label' : 'Disaster or Individual Tracking', 'value' : 'Disaster'}
            ],
            value='All' #default value
        )
    ]),
    html.Hr(),
    dash_table.DataTable(id='datatable-id',
                         columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
                         data=df.to_dict('records'),
#FIXME: Set up the features for your interactive data table to make it user-friendly for your client
#If you completed the Module Six Assignment, you can copy in the code you created here 
                         editable=False, # used for testing
        row_selectable="single", #copied from my Module Six Assignment
        selected_rows=[],
        filter_action="native", # sets to native to match native
        sort_action="native",
        page_action="native",
        page_current=0,
        page_size=10,
                         style_data_conditional=[] # was getting an error until I added this to define empty list.
                        ),
    html.Br(),
    html.Hr(),
#This sets up the dashboard so that your chart and your geolocation chart are side-by-side
    html.Div(className='row',
         style={'display' : 'flex'},
             children=[
        html.Div(
            id='graph-id',
            className='col s12 m6',

            ),
        html.Div(
            id='map-id',
            className='col s12 m6',
            )
        ])
])

#############################################
# Interaction Between Components / Controller
#############################################



    
@app.callback([Output('datatable-id','data'),
              Output('datatable-id', 'columns')], # added columns for a second retunred type
              [Input('filter-type', 'value')])
def update_dashboard(filter_type):
## FIX ME Add code to filter interactive data table with MongoDB queries

    if filter_type == 'All': #returns all querable animals
        query = {}
    elif filter_type == 'Water':
        query = {
            'animal_type': 'Dog',
            '$or': [ # I used regex to parse through the animals and pull the ones matching the correct "description" because the data seems a little messy
                {"breed": {'$regex': re.compile(".*lab.*", re.IGNORECASE)}},
                    {"breed": {'$regex': re.compile(".*chesa.*", re.IGNORECASE)}},                   
                {"breed": {'$regex': re.compile(".*newf.*", re.IGNORECASE)}},
            ],
            'sex_upon_outcome': 'Intact Female',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156                
            }
        }
    elif filter_type == "Mountain":
        query = {
            'animal_type': 'Dog',
            '$or': [
                {"breed": {'$regex': re.compile(".*german.*", re.IGNORECASE)}},
                {"breed": {'$regex': re.compile(".*mala.*", re.IGNORECASE)}},                   
                {"breed": {'$regex': re.compile(".*husk.*", re.IGNORECASE)}},
                {"breed": {'$regex': re.compile(".*rott.*", re.IGNORECASE)}},                   
                {"breed": {'$regex': re.compile(".*old english.*", re.IGNORECASE)}},
            ],
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 26,
                '$lte': 156
            }
        }
    elif filter_type == "Disaster":
        query = {
            'animal_type': 'Dog',
            '$or': [
                    {"breed": {'$regex': re.compile(".*german.*", re.IGNORECASE)}},
                    {"breed": {'$regex': re.compile(".*golden.*", re.IGNORECASE)}},                   
                    {"breed": {'$regex': re.compile(".*blood.*", re.IGNORECASE)}},
                    {"breed": {'$regex': re.compile(".*dober.*", re.IGNORECASE)}},                   
                    {"breed": {'$regex': re.compile(".*rott.*", re.IGNORECASE)}},
                ],
            'sex_upon_outcome': 'Intact Male',
            'age_upon_outcome_in_weeks': {
                '$gte': 20,
                '$lte': 300
            }
        }
    else:
        raise Exception("Unknown filter")
                    
    results = db.read(query)
        
    df = pd.DataFrame.from_records(results)
    
    if '_id' in df.columns: # drops empty ids
        df.drop(columns=['_id'], inplace=True)
        
    columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns]
    data=df.to_dict('records')
      
      
    return data, columns

# Display the breeds of animal based on quantity represented in
# the data table


@app.callback(
    Output('graph-id', "children"),
    [Input('datatable-id', "derived_virtual_data")])
def update_graphs(viewData):
    ###FIX ME ####
    # add code for chart of your choice (e.g. pie chart) #
    if not viewData:
        return html.P('No animals found')
    
    dffPie = pd.DataFrame.from_dict(viewData)
    
    df = (
        dffPie['breed']
        .fillna('unknown')
        .value_counts()
        .head(10)
        .reset_index()
    )
    
    df.columns = ['Breed', 'Count']
    
    return [
       dcc.Graph(            
           figure = px.pie(df, names='Breed', values='Count', title='Preferred Animals') # returns the graph of breeds, with counts
       )    
    ]
    
#This callback will highlight a cell on the data table when the user selects it
@app.callback(
    Output('datatable-id', 'style_data_conditional'),
    [Input('datatable-id', 'selected_columns')]
)
def update_styles(selected_columns):
    if not selected_columns:
        return []
    return [{
        'if': { 'column_id': i },
        'background_color': '#D2F3FF'
    } for i in selected_columns]


# This callback will update the geo-location chart for the selected data entry
# derived_virtual_data will be the set of data available from the datatable in the form of 
# a dictionary.
# derived_virtual_selected_rows will be the selected row(s) in the table in the form of
# a list. For this application, we are only permitting single row selection so there is only
# one value in the list.
# The iloc method allows for a row, column notation to pull data from the datatable
@app.callback(
    Output('map-id', "children"),
    [Input('datatable-id', "derived_virtual_data"),
     Input('datatable-id', "derived_virtual_selected_rows")])
def update_map(viewData, index):
#FIXME Add in the code for your geolocation chart

    if viewData is None or len(viewData) == 0: # returns nothing if there are no animals to display
        return html.P(' No animals available to display') 
    
    dff = pd.DataFrame.from_dict(viewData)
    if index is None or len(index) == 0: # would not create the map if the len(index) == 0, and caused an error where the children list index out of range
        row = 0
    else:
        row = index[0]
        
    animal = dff.iloc[row] # animal
        
    latitude = pd.to_numeric(animal.get('location_lat'), errors="coerce") # pulled it out to make more managable, uses animal object to find location
    longitude = pd.to_numeric(animal.get('location_long'), errors="coerce")
    
    breed = animal.get('breed', 'Unknown breed') # finds the breed of the animal
    animal_name = animal.get('name', '') # finds the name of the animal
    animal_id = animal.get('animal_id', 'Unknown ID') # finds the ID of the animal
    
    if pd.isna(latitude) or pd.isna(longitude):
        return html.P("The selected animal has no valid coordinates")
    
    if pd.isna(animal_name) or not str(animal_name).strip():
        animal_name = animal_id # In Module Six, I had it return nothing if there was no animal name. Probably better to just default to ID for identification purposes
    
    return [
        dl.Map(
            style={"width": "1000px", "height": "500px"}, # from the post
            center=[latitude, longitude], # goes to direct location of animal, better in general
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                
                dl.Marker(
                    position=[latitude, longitude], # works similarily
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H3(str(animal_name)), # returns animal name (id if there is no possible name)
                            html.P('Breed: ' + str(breed)),
                            html.P('Animal ID:' + str(animal_id))
                        ])
                ])
        ])
    ]

# Run app and display result in jupyterlab mode, note, if you have previously run a prior app, the default port of 8050 may not be available, if so, try setting an alternate port.
app.run_server() 

Dash app running on https://decimalstamp-athletehello-3000.codio.io/proxy/8050/
